In [0]:
bookings_df = spark.table("bookings")
facilities_df = spark.table("facilities")

from pyspark.sql.functions import month, year, sum

bookings_facilities_df = bookings_df.join(facilities_df, bookings_df['facid'] == facilities_df['facid'])

aggregated_df = bookings_facilities_df.withColumn("year", year("starttime")).withColumn("month", month("starttime")).groupBy("year", "month", "name").agg(sum("slots").alias("total_hours")).orderBy("year", "month", "name")

aggregated_df.show()

output_path = "/tmp/output/facilities_usage_by_month.parquet"

aggregated_df.write.mode("overwrite").parquet(output_path)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-89251925615179>:1
----> 1 bookings_df = spark.table("bookings")

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/sql/session.py:1423, in SparkSession.table(self, tableName)
   1392 def table(self, tableName: str) -> DataFrame:
   1393     """Returns the specified table as a :class:`DataFrame`.
   1394 
   1395     .. versionadded:: 2.0.0
   (...)
   1421     +---+
   1422     """
-> 1423     return DataFrame(self._jsparkSession.table(tableName), self)

File /databric

In [0]:
bookings_df = spark.table("bookings")
members_df = spark.table("members")
facilities_df = spark.table("facilities")

joined_df = bookings_df.join(members_df, bookings_df['memid'] == members_df['memid']).join(facilities_df, bookings_df['facid'] == facilities_df['facid'])

transformed_df = joined_df.select("facilities.name", "members.surname", "bookings.starttime")

transformed_df.write.mode("overwrite").partitionBy("facid").format("delta") .saveAsTable("threejoin_delta")

In [0]:
# Extract
import requests

def fetch_stock_data(symbol):
    url = "https://alpha-vantage.p.rapidapi.com/query"
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "datatype": "json",
        "outputsize": "compact"
    }
    headers = {
        "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
        "X-RapidAPI-Key": "[YOUR_KEY]"
    }
    
    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()
    return data

companies = ["GOOGL", "AAPL", "MSFT", "TSLA"]

stock_data = {}
for company in companies:
    stock_data[company] = fetch_stock_data(company)

# Transform
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max, weekofyear, to_date

spark = SparkSession.builder.getOrCreate()

def transform_data(data, company):
    daily_data = data['Time Series (Daily)']

    rows = [(date, daily_data[date]['4. close']) for date in daily_data]

    df = spark.createDataFrame(rows, ["date", "closing_price"])

    df = df.withColumn("company", lit(company)) .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) .withColumn("week", weekofyear(col("date")))
    
    return df

combined_df = None

for company in stock_data:
    company_df = transform_data(stock_data[company], company)
    
    if combined_df is None:
        combined_df = company_df
    else:
        combined_df = combined_df.union(company_df)

max_closing_df = combined_df.groupBy("company", "week").agg(max("closing_price").alias("max_closing_price"))

max_closing_df.show()

# Load
max_closing_df.write.mode("overwrite").partitionBy("company").format("delta").saveAsTable("max_closing_price_weekly")

In [0]:
jdbc_url = "jdbc:postgresql://localhost:5432/pgexercise"
table = "rna"
properties = {
    "user": "username",
    "password": "password",
    "driver": "org.postgresql.Driver"
}

query = "(SELECT * FROM rna LIMIT 100) AS rna_subset"

rna_df = spark.read.jdbc(url=jdbc_url, table=query, properties=properties)

rna_df.write.mode("overwrite").saveAsTable("rna_100_records")